<a href="https://colab.research.google.com/github/umair594/Remote-internship_DS_CodeAlpha/blob/main/Handwritten_Character_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Handwritten Character Recognition**

# **Project Overview**

**Goal:**

Classify handwritten characters (digits or alphabets) from images.

**Core stack:**

Python

TensorFlow / Keras (or PyTorch alternative)

OpenCV (optional preprocessing)

NumPy / Matplotlib

**Datasets:**

MNIST → digits (0–9)

EMNIST → letters + digits

# **Install Dependencies**

In [ ]:
pip install tensorflow numpy matplotlib opencv-python scikit-learn

# **Data Loading (MNIST / EMNIST)**

**MNIST Example (built-in)**

In [2]:
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


# **Preprocessing**

In [3]:
# Normalize
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape for CNN
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# **CNN Model (Core)**

In [4]:
from tensorflow.keras import layers, models

def build_model():
    model = models.Sequential()

    model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(64, (3,3), activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(128, (3,3), activation='relu'))

    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))

    model.add(layers.Dense(10, activation='softmax'))  # 10 classes (digits)

    return model

# **Training**

In [5]:
model = build_model()

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    x_train, y_train,
    epochs=10,
    validation_data=(x_test, y_test),
    batch_size=64
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 57s 58ms/step - accuracy: 0.9313 - loss: 0.2223 - val_accuracy: 0.9809 - val_loss: 0.0602
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 83s 59ms/step - accuracy: 0.9814 - loss: 0.0647 - val_accuracy: 0.9902 - val_loss: 0.0304
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 80s 58ms/step - accuracy: 0.9865 - loss: 0.0456 - val_accuracy: 0.9898 - val_loss: 0.0305
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 55s 59ms/step - accuracy: 0.9895 - loss: 0.0354 - val_accuracy: 0.9912 - val_loss: 0.0264
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 55s 59ms/step - accuracy: 0.9912 - loss: 0.0301 - val_accuracy: 0.9915 - val_loss: 0.0289
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 56s 60ms/step - accuracy: 0.9920 - loss: 0.0254 - val_accuracy: 0.9928 - val_loss: 0.0252
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 53s 56ms/step - accuracy: 0.9932 - loss: 0.0216 - val_accuracy: 0.9911 - val_loss: 0.0300
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 52s 56ms/step - accuracy: 0.9946 - loss: 0.0176 - 

# **Evaluation**

In [7]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9910 - loss: 0.0353
Accuracy: 0.9909999966621399


**Expected:**

MNIST accuracy: **~99%**

# **Save & Load Model**

In [8]:
model.save("models/handwritten_model.h5")

# Load later
from tensorflow.keras.models import load_model
model = load_model("models/handwritten_model.h5")

# **Prediction on Custom Image**

In [10]:
import cv2
import numpy as np

def predict_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (28, 28))
    img = img / 255.0
    img = img.reshape(1, 28, 28, 1)

    prediction = model.predict(img)
    return np.argmax(prediction)

# Create a dummy image for prediction if it doesn't exist
# We'll use the first image from x_test for this example
if not cv2.imread("digit.png", cv2.IMREAD_GRAYSCALE) is None:
    print("Using existing digit.png")
else:
    print("Creating digit.png from x_test[0]")
    # Convert normalized float to uint8 for saving, scaling by 255
    sample_image = (x_test[0].reshape(28, 28) * 255).astype(np.uint8)
    cv2.imwrite("digit.png", sample_image)

print("Predicted:", predict_image("digit.png"))

Creating digit.png from x_test[0]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Predicted: 7


# **Extend to EMNIST (Alphabets)**

In [11]:
model.add(layers.Dense(10, activation='softmax'))

**With:**

In [12]:
model.add(layers.Dense(62, activation='softmax'))  # letters + digits

# **Advanced Extension: Word Recognition (CRNN)**

**To scale from characters → words:**

**Architecture:**

CNN → feature extraction

RNN (LSTM/GRU) → sequence modeling

CTC Loss → alignment

**Concept Pipeline:**

**Image → CNN → Feature Map → Reshape → LSTM → Dense → CTC Decoder**

**Tools:**

TensorFlow + Keras CTC

PyTorch + CRNN

# **Optional Enhancements**

**Data Augmentation**

In [13]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

datagen.fit(x_train)

**Real-Time Drawing App (Optional)**

**Use:**

OpenCV canvas

OR

Streamlit UI

# **Performance Improvements**

Batch Normalization

Deeper CNN (ResNet-like)

Transfer learning (for complex handwriting)

Hyperparameter tuning

# **Deliverables (for submission)**

**Include:**

Source code

Trained model (.h5)

README.md

Sample predictions

Accuracy graphs